## Load NAB time series

## Step 1 - Load NYC taxi dataset and attach anomaly labels
In this step, the raw `nyc_taxi.csv` file from the NAB `realKnownCause` folder is loaded into a
dataframe. The corresponding anomaly timestamps for this dataset are retrieved from
`combined_labels.json` using the key `realKnownCause/nyc_taxi.csv` and converted into a binary
`is_anomaly` column (0 = normal, 1 = labelled anomaly). The `timestamp` column is parsed as a
proper datetime type and the rows are sorted chronologically so that the NYC taxi demand series is
cleanly aligned in time and each anomaly label matches the correct 30-minute interval.


### A) Load raw nyc_taxi.csv dataset

In [7]:
# Imports
import pandas as pd
import matplotlib.pyplot as plt

In [8]:
import os

# Path to Numeta Anomoly Benchmark (NAB) data folder
base_path = "../data/NAB-master/data/realKnownCause/"

# Load one example file
file_path = os.path.join(base_path, "nyc_taxi.csv")
df = pd.read_csv(file_path)

df.head()


,timestamp,value
0,2014-07-01 00:00:00,10844
1,2014-07-01 00:30:00,8127
2,2014-07-01 01:00:00,6210
3,2014-07-01 01:30:00,4656
4,2014-07-01 02:00:00,3820


### B) Load Anomaly Labels (JSON)

In [9]:
import json  # to work with JSON files

# Tell Python where the labels file is
label_path = "../data/NAB-master/labels/combined_labels.json"

# Open the file and read its contents
with open(label_path, "r") as label_file:      # "r" = read mode, label_file = name for the opened file
    labels = json.load(label_file)             # turn the JSON into a Python dictionary called 'labels'

# Shows the first 10 dataset keys names to confirm it loaded correctly
list(labels.keys())[:10]



['artificialNoAnomaly/art_daily_no_noise.csv',
 'artificialNoAnomaly/art_daily_perfect_square_wave.csv',
 'artificialNoAnomaly/art_daily_small_noise.csv',
 'artificialNoAnomaly/art_flatline.csv',
 'artificialNoAnomaly/art_noisy.csv',
 'artificialWithAnomaly/art_daily_flatmiddle.csv',
 'artificialWithAnomaly/art_daily_jumpsdown.csv',
 'artificialWithAnomaly/art_daily_jumpsup.csv',
 'artificialWithAnomaly/art_daily_nojump.csv',
 'artificialWithAnomaly/art_increase_spike_density.csv']

### C) Extract Labels for Selected Dataset

In [10]:
# Pick out the labels for the NYC taxi dataset
dataset_key = "realKnownCause/nyc_taxi.csv"

nyc_taxi_labels = labels[dataset_key]

# Turn the list of timestamp strings into a DataFrame
nyc_taxi_labels_df = pd.DataFrame(
    {"timestamp": pd.to_datetime(nyc_taxi_labels)}
)

nyc_taxi_labels_df.head()

,timestamp
0,2014-11-01 19:00:00
1,2014-11-27 15:30:00
2,2014-12-25 15:00:00
3,2015-01-01 01:00:00
4,2015-01-27 00:00:00


### D) Add Labels to DataFrame

In [11]:
# Ensure main dataframe timestamps are in datetime and sorted
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

# Convert the anomaly label timestamps to datetime
# use the list nyc_taxi_labels directly
anomaly_times = pd.to_datetime(nyc_taxi_labels)

# Create is_anomaly: 1 if this timestamp is labelled, else 0
df["is_anomaly"] = df["timestamp"].isin(anomaly_times).astype(int)

# Quick sanity check
df[["timestamp", "value", "is_anomaly"]].head()


,timestamp,value,is_anomaly
0,2014-07-01 00:00:00,10844,0
1,2014-07-01 00:30:00,8127,0
2,2014-07-01 01:00:00,6210,0
3,2014-07-01 01:30:00,4656,0
4,2014-07-01 02:00:00,3820,0


### E) Class Balance

In [12]:
# Class balance for the is_anomaly column
anomaly_balance = (
    df["is_anomaly"]
      .value_counts()                            # counts for 0 and 1
      .rename(index={0: "Normal", 1: "Anomaly"}) # rename classes for readability
      .to_frame(name="Count")                    # make it a DataFrame with a 'Count' column
      .reset_index()                             # turn index into a normal column
      .rename(columns={"is_anomaly": "Class"})       
)

# Percentage column
anomaly_balance["Proportion (%)"] = (
    anomaly_balance["Count"] / len(df) * 100
).round(3)

anomaly_balance


,Class,Count,Proportion (%)
0,Normal,10315,99.952
1,Anomaly,5,0.048


## Step 2 - Standardise core columns (`time`, `value`, `is_anomaly`, `case_study`)

In [13]:
# Ensure timestamp is datetime and sorted
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

# Rename to common research project schema
df = df.rename(columns={"timestamp": "time"})

# Add case_study identifier
df["case_study"] = "nyc_taxi"

# Quick check
df = df[["time", "value", "is_anomaly", "case_study"]]

df.head()


,time,value,is_anomaly,case_study
0,2014-07-01 00:00:00,10844,0,nyc_taxi
1,2014-07-01 00:30:00,8127,0,nyc_taxi
2,2014-07-01 01:00:00,6210,0,nyc_taxi
3,2014-07-01 01:30:00,4656,0,nyc_taxi
4,2014-07-01 02:00:00,3820,0,nyc_taxi


###  Quick verification

In [14]:
df[["time","value","is_anomaly","case_study"]].dtypes

time          datetime64[ns]
value                  int64
is_anomaly             int64
case_study            object
dtype: object

In this step, the NYC taxi dataframe is aligned with the common schema used across all case
studies. The `timestamp` column is converted to a proper datetime type, renamed to `time`, and the
rows are sorted chronologically. A `case_study` column with the value `"nyc_taxi"` is added so that
this dataset can later be stacked or filtered alongside other case studies using the same column
names (`time`, `value`, `is_anomaly`, `case_study`).


## Step 3 - Missing values and Duplicates rows sanity check

### A) Missing values per column

In [8]:
# Missing values per column
missing_counts = df.isna().sum()

# Percentage of missing values in each column
missing_percentages = (df.isna().mean() * 100).round(3)

# Combine into a summary table
missing_summary = (
    pd.DataFrame(
        {
            "Column": missing_counts.index,
            "Missing count": missing_counts.values,
            "Missing (%)": missing_percentages.values,
        }
    )
    .sort_values("Missing count", ascending=False)
    .reset_index(drop=True)
)

missing_summary


,Column,Missing count,Missing (%)
0,time,0,0.0
1,value,0,0.0
2,is_anomaly,0,0.0
3,case_study,0,0.0


### B) Duplicate rows summary

In [9]:
# B) Duplicate rows summary
duplicate_count = df.duplicated().sum()
total_rows = len(df)

duplicate_summary = pd.DataFrame(
    [
        ("Total rows", total_rows),
        ("Number of duplicate rows", duplicate_count),
        ("Duplicate rows (%)", round(duplicate_count / total_rows * 100, 3)),
    ],
    columns=["Statistic", "Value"],
)

duplicate_summary


,Statistic,Value
0,Total rows,10320.0
1,Number of duplicate rows,0.0
2,Duplicate rows (%),0.0


The missing-values table shows that the NYC taxi dataset has no missing values in any of the core
columns (`time`, `value`, `is_anomaly`, `case_study`). The duplicate-rows summary likewise reports
zero duplicate records. This confirms that the NAB NYC taxi series is clean at the basic data
quality level, so no additional imputation or de-duplication is needed. Documenting these negative
findings keeps the preprocessing steps transparent and easy to justify in the methodology chapter.


## Step 4 - Create basic time-based features

In [10]:
# Hour of day (0–23)
df["hour_of_day"] = df["time"].dt.hour

# Day of week (0 = Monday, 6 = Sunday)
df["day_of_week"] = df["time"].dt.dayofweek

# Weekend flag (Saturday = 5, Sunday = 6)
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

# Quick check
df[["time", "hour_of_day", "day_of_week", "is_weekend"]].head()


,time,hour_of_day,day_of_week,is_weekend
0,2014-07-01 00:00:00,0,1,0
1,2014-07-01 00:30:00,0,1,0
2,2014-07-01 01:00:00,1,1,0
3,2014-07-01 01:30:00,1,1,0
4,2014-07-01 02:00:00,2,1,0


This step derives simple calendar features from the `time` column to provide
additional structure for later analysis and modelling. The timestamp is broken
down into:

- `hour_of_day`: the hour of the day (0–23),
- `day_of_week`: the day index (0 = Monday, …, 6 = Sunday),
- `is_weekend`: an indicator variable that is 1 for Saturday and Sunday, and 0
  for weekdays.

These features do not change the underlying temperature values, but they allow
later models and visualisations to capture patterns that depend on time of day
or weekday/weekend effects.


## Step 5 - Inspect time span and anomaly distribution

### 5A) Anomaly timestamps and values

In [11]:
# Filter to rows labelled as anomalies and keep only columns time and value
anomaly_summary = (
    df.loc[df["is_anomaly"] == 1, ["time", "value"]]
      .sort_values("time")
      .reset_index(drop=True)
)

# Insterting an Anomaly ID column for readability
anomaly_summary.insert(0, "Anomaly ID", [f"Anomaly {i+1}" for i in range(len(anomaly_summary))])

anomaly_summary


,Anomaly ID,time,value
0,Anomaly 1,2014-11-01 19:00:00,28398
1,Anomaly 2,2014-11-27 15:30:00,15255
2,Anomaly 3,2014-12-25 15:00:00,12039
3,Anomaly 4,2015-01-01 01:00:00,30236
4,Anomaly 5,2015-01-27 00:00:00,109


### 5B) Overall time span summary

In [12]:
# Time-span statistics for the series
start_time, end_time = df["time"].min(), df["time"].max()
n_rows = len(df)
duration_days = (end_time - start_time).days

# Summary table 
time_span_summary = pd.DataFrame(
    [
        ("Start time", start_time),
        ("End time", end_time),
        ("Number of rows", n_rows),
        ("Total duration (days)", duration_days),
    ],
    columns=["Statistic", "Value"],
)

time_span_summary


,Statistic,Value
0,Start time,2014-07-01 00:00:00
1,End time,2015-01-31 23:30:00
2,Number of rows,10320
3,Total duration (days),214


Step 5 summarises where the labelled anomalies fall within the NYC taxi series and how long the
series runs overall.

The anomaly table lists the five NAB-labelled events with an `Anomaly ID`, the exact 30-minute
timestamp, and the corresponding trip count. These entries provide concrete reference points for
describing unusual demand spikes (e.g. public holidays or disruptions) and for later defining
evaluation windows around each anomaly.

The time-span summary shows that the NYC taxi dataset runs from 2014-07-01 00:00:00 to
2015-01-31 23:30:00, with 10 320 half-hourly observations covering a total of 214 days. This
information is useful for briefly characterising the dataset in the methodology and results
chapters, and it provides context for how the chronological train, validation, and test splits will
be defined in the next steps.


## Step 6 - Define chronological train/validation/test boundaries

### A) Boundery selection

In [13]:
# Basic data range for context
dataset_start = df["time"].min()
dataset_end = df["time"].max()

# Choose boundaries based on calendar months:
#  Train:       2014-07-01 to 2014-10-31 23:30 (July-Oct)
#  Validation:  2014-11-01 to 2014-12-31 23:30 (Nov-Dec)
#  Test:        2015-01-01 onwards (Jan)
training_end_time = pd.Timestamp("2014-10-31 23:30:00")
validation_end_time = pd.Timestamp("2014-12-31 23:30:00")

# Summary table of chosen boundaries
boundary_summary = pd.DataFrame(
    [
        ("Dataset start ", dataset_start),
        ("Training end time", training_end_time),
        ("Validation end time", validation_end_time),
        ("Dataset end ", dataset_end),
    ],
    columns=["Description", "Time"],
)

boundary_summary


,Description,Time
0,Dataset start,2014-07-01 00:00:00
1,Training end time,2014-10-31 23:30:00
2,Validation end time,2014-12-31 23:30:00
3,Dataset end,2015-01-31 23:30:00


This step records the overall time range of the NYC taxi series and sets explicit calendar-based
boundaries for the train, validation, and test segments. The dataset runs from 2014-07-01 to
2015-01-31 (July - January). 
To keep the splits easy to interpret and aligned with natural periods, the series is
divided by month:

- Training covers July to the end of October 2014.
- Validation covers November to the end of December 2014.
- Test covers January 2015 as the most recent month.

These boundaries respect the time order of the data (no future information leaks into training) and
create a clear narrative: models are trained on several months of “typical” demand, tuned on later
months that include labelled unusual events, and finally evaluated on the last month of the series.
The table above records the dataset start and end times together with the chosen boundary times for
later reference in the methodology chapter.


### B) Preview of where anomalies fall in the defined bounderies (train, validation, test)

In [14]:
anomaly_split = anomaly_summary.copy()

# Assign proposed split label based on the boundary times
def preview_split(time_value):
    if time_value <= training_end_time:
        return "train"
    elif time_value <= validation_end_time:
        return "validation"
    else:
        return "test"

anomaly_split["Proposed split"] = anomaly_split["time"].apply(preview_split)

anomaly_split


,Anomaly ID,time,value,Proposed split
0,Anomaly 1,2014-11-01 19:00:00,28398,validation
1,Anomaly 2,2014-11-27 15:30:00,15255,validation
2,Anomaly 3,2014-12-25 15:00:00,12039,validation
3,Anomaly 4,2015-01-01 01:00:00,30236,test
4,Anomaly 5,2015-01-27 00:00:00,109,test


Here, the labelled anomalies are mapped to the proposed train/validation/test boundaries to check
where each event will fall once the splits are applied. For each anomaly timestamp, a provisional
split label is assigned based on the boundary times defined in Step 6A.

This preview serves two purposes:

- It confirms that the training period does not unintentionally include labelled anomaly events,
  keeping training focused on normal behaviour.
- It shows that the validation and test segments each contain anomaly examples, which is important
  for model selection and final evaluation in a rare-event setting.

If the preview showed anomalies in the wrong segment (for example, all anomalies ending up in
validation and none in test), the boundaries could be adjusted at this stage before creating the
final `split` column for all rows.


## Step 7 - Assign split labels to all rows

### A) Assign split labels to all rows based on the boundary times

In [15]:
# Helper function to map a single timestamp to a split label
def assign_split(time_value):
    if time_value <= training_end_time:
        return "train"
    elif time_value <= validation_end_time:
        return "validation"
    else:
        return "test"

# Apply the function to the time column to create the split labels
df["split"] = df["time"].apply(assign_split)

# Check of the first few rows
df[["time", "value", "is_anomaly", "split"]].head()


,time,value,is_anomaly,split
0,2014-07-01 00:00:00,10844,0,train
1,2014-07-01 00:30:00,8127,0,train
2,2014-07-01 01:00:00,6210,0,train
3,2014-07-01 01:30:00,4656,0,train
4,2014-07-01 02:00:00,3820,0,train


### 7B) Summarise the train / validation / test segments


In [16]:
# Total rows in the full dataframe 
total_rows = len(df)

# Aggregate key information per split
split_counts = (
    df.groupby("split")
      .agg(
          start_time=("time", "min"),        
          end_time=("time", "max"),          
          Rows=("time", "count"),            
          Anomalies=("is_anomaly", "sum"),   
      )
      .reset_index()
)

# Proportion of total rows as a percentage
split_counts["Proportion (%)"] = (split_counts["Rows"] / total_rows * 100).round(3)

# Enforce normal split order
split_order = pd.CategoricalDtype(["train", "validation", "test"], ordered=True)
split_counts["split"] = split_counts["split"].astype(split_order)
split_counts = split_counts.sort_values("split").reset_index(drop=True)

# Format start and end times as full date-time strings for consistent display
split_counts["Start time"] = split_counts["start_time"].dt.strftime("%Y-%m-%d %H:%M:%S")
split_counts["End time"]   = split_counts["end_time"].dt.strftime("%Y-%m-%d %H:%M:%S")

# Build final examiner-style table with clean column order
split_summary = split_counts[
    ["split", "Start time", "End time", "Rows", "Anomalies", "Proportion (%)"]
]

split_summary


,split,Start time,End time,Rows,Anomalies,Proportion (%)
0,train,2014-07-01 00:00:00,2014-10-31 23:30:00,5904,0,57.209
1,validation,2014-11-01 00:00:00,2014-12-31 23:30:00,2928,3,28.372
2,test,2015-01-01 00:00:00,2015-01-31 23:30:00,1488,2,14.419


Step 7 assigns a split label (`train`, `validation`, `test`) to every row in the NYC taxi dataset
using the calendar-based boundaries defined earlier, and then summarises the resulting segments.

The `split` column is created by comparing each timestamp to the chosen boundary times
(2014-10-31 23:30 and 2014-12-31 23:30). All observations up to the end of October are labelled
`train`, November–December observations are labelled `validation`, and January 2015 observations are
labelled `test`. This enforces a strictly chronological design where no future information leaks
into the training period.

The summary table shows, for each split, the start and end times, number of rows, anomaly counts,
and the proportion of the full dataset. In this configuration, training covers the first four
months with 0 anomalies (normal behaviour only), validation covers November–December with several
labelled anomalies for model tuning, and test covers January with the remaining anomalies for final
evaluation. This distribution is appropriate for a rare-event setting and provides a clear,
examiner-friendly justification for later modelling and comparison across case studies.


## Step 8 - Create split-specific dataframes and Integrity check

In [17]:
# Keep track of the total number of rows in the full processed NYC taxi dataframe
total_rows_full = len(df)

# Create non-destructive split-specific dataframes using boolean masks
train_df = df[df["split"] == "train"].copy()
validation_df = df[df["split"] == "validation"].copy()
test_df = df[df["split"] == "test"].copy()

# Row counts in each split dataframe
n_train = len(train_df)
n_validation = len(validation_df)
n_test = len(test_df)

# Summary to check that no rows are lost
split_integrity_summary = pd.DataFrame(
    [
        ("Total rows in full df", total_rows_full),
        ("Rows in train_df", n_train),
        ("Rows in validation_df", n_validation),
        ("Rows in test_df", n_test),
        ("Sum of split rows", n_train + n_validation + n_test),
    ],
    columns=["Statistic", "Value"],
)

split_integrity_summary


,Statistic,Value
0,Total rows in full df,10320
1,Rows in train_df,5904
2,Rows in validation_df,2928
3,Rows in test_df,1488
4,Sum of split rows,10320


## Step 9 - Standardise value using training split

### 9A) Fit a StandardScaler on the NYC taxi training values and store parameters

In [18]:
from sklearn.preprocessing import StandardScaler

# Initialise the scaler
scaler = StandardScaler()

# Fit the scaler using ONLY the training split (raw trip counts)
scaler.fit(train_df[["value"]])

# Store the training mean and standard deviation for documentation/reproducibility
nyc_taxi_scaler_parameters = {
    "mean": float(scaler.mean_[0]),
    "std": float(scaler.scale_[0]),
}

nyc_taxi_scaler_parameters


{'mean': 15318.730521680216, 'std': 6808.285657581778}

### 9B) Apply scaling to full dataframe and check training behaviour

In [20]:
# Apply the fitted scaler to the FULL NYC taxi dataframe (train, validation, test)
df["value_scaled"] = scaler.transform(df[["value"]])

# Describe the scaled values on the training split to confirm mean ~ 0 and std ~ 1
scaled_training_summary = (
    df[df["split"] == "train"]["value_scaled"]
      .describe()[["mean", "std"]]
      .to_frame(name="Training value_scaled")
      .rename(index={"mean": "Mean", "std": "Std. deviation"})
      .round(4)
)

scaled_training_summary

,Training value_scaled
Mean,0.0000
Std. deviation,1.0001


### 9C) Summary of value_scaled by split

In [23]:
# Summary of scaled values per split
split_scaled_summary = (
    df.groupby("split")["value_scaled"]
      .describe()[["mean", "std", "min", "max"]]   # key statistics only
      .round(3)
      .reset_index()
)

split_scaled_summary


,split,mean,std,min,max
0,test,-0.135,1.077,-2.249,2.191
1,train,0.000,1.000,-2.040,2.211
2,validation,-0.025,1.024,-2.036,3.507


### Standardising the NYC taxi demand values

- **Cell 9A – Fit the scaler on the training split and store parameters**
  - Fits a `StandardScaler` using only the `value` column from `train_df`.
  - Learns the average trip count (mean) and spread (standard deviation) of the training data.
  - Saves these numbers in `nyc_taxi_scaler_parameters = {"mean": ..., "std": ...}` so the scaling
    step is transparent and can be reported or reproduced later.
  - Respects the rule that all preprocessing statistics are derived from the training split only,
    avoiding information leakage from validation/test into training.

- **Cell 9B – Apply scaling to the full dataframe and check training behaviour**
  - Uses the fitted scaler to create a new column `value_scaled` for all rows in `df` (train,
    validation, and test).
  - Checks that, on the training split, `value_scaled` has mean ≈ 0 and standard deviation ≈ 1.
  - Confirms that the models will see a demand feature that is on a standard scale, which helps
    optimisation and makes it easier to compare behaviour across case studies.

- **Cell 9C – Summary of `value_scaled` by split (train / validation / test)**
  - Groups the data by `split` and reports mean, standard deviation, minimum, and maximum of
    `value_scaled` for each segment.
  - Shows how the scaled demand distribution differs between training, validation, and test
    periods, which is useful for understanding any drift or regime changes over time.
  - Provides a compact table that documents the effect of scaling across all three splits for the
    NYC taxi dataset.


## Reorder columns for a clean schema

In [25]:
column_order = [
    "time",
    "value",
    "value_scaled",
    "is_anomaly",
    "hour_of_day",
    "day_of_week",
    "is_weekend",
    "split",
    "case_study",
]

# Reorder the dataframe using the defined schema
df = df[column_order]

# Quick check of the first few rows
df.head()



,time,value,value_scaled,is_anomaly,hour_of_day,day_of_week,is_weekend,split,case_study
0,2014-07-01 00:00:00,10844,-0.657248,0,0,1,0,train,nyc_taxi
1,2014-07-01 00:30:00,8127,-1.056320,0,0,1,0,train,nyc_taxi
2,2014-07-01 01:00:00,6210,-1.337889,0,1,1,0,train,nyc_taxi
3,2014-07-01 01:30:00,4656,-1.566140,0,1,1,0,train,nyc_taxi
4,2014-07-01 02:00:00,3820,-1.688932,0,2,1,0,train,nyc_taxi


## Recreate split-specific dataframes so they include `value_scaled`

In [28]:
train_df = df[df["split"] == "train"].copy()
validation_df = df[df["split"] == "validation"].copy()
test_df = df[df["split"] == "test"].copy()


In [29]:
#Quick check, rows should be equal
len(df), len(train_df) + len(validation_df) + len(test_df)

(10320, 10320)

## Step 10 – Save processed ambient datasets to disk

In [38]:
import os

# Folder for processed NYC taxi data 
output_dir = "../data/processed/nyc_taxi"
os.makedirs(output_dir, exist_ok=True)

# Save full processed dataframe (ground truth processed file)
full_path = os.path.join(output_dir, "nyc_taxi_full.csv")
df.to_csv(full_path, index=False)

# File paths for the three splits
train_path = os.path.join(output_dir, "nyc_taxi_train.csv")
val_path   = os.path.join(output_dir, "nyc_taxi_val.csv")
test_path  = os.path.join(output_dir, "nyc_taxi_test.csv")

# Save each split without the index column
train_df.to_csv(train_path, index=False)
validation_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)

train_path, val_path, test_path, full_path


('../data/processed/nyc_taxi/nyc_taxi_train.csv',
 '../data/processed/nyc_taxi/nyc_taxi_val.csv',
 '../data/processed/nyc_taxi/nyc_taxi_test.csv',
 '../data/processed/nyc_taxi/nyc_taxi_full.csv')

## Save processed ambient datasets

The following CSV files are written to the `data/processed/nyc_taxi` folder:

- `nyc_taxi_full.csv`  
  - Contains the entire processed NYC taxi dataset after preprocessing.  
  - Includes all rows and core columns:  
    `time`, `value`, `value_scaled`, `is_anomaly`, `hour_of_day`, `day_of_week`, `is_weekend`, `split`, `case_study`.  
  - This is the canonical reference file for NYC taxi and the starting point for any stacked,
    cross–case study views.

- `nyc_taxi_train.csv`  
  - Subset of `nyc_taxi_full.csv` where `split == "train"`.  
  - Used for fitting models and computing preprocessing statistics (e.g. scaling parameters).

- `nyc_taxi_val.csv`  
  - Subset of `nyc_taxi_full.csv` where `split == "validation"`.  
  - Used for model selection, hyperparameter tuning, and early comparison of baselines vs
    diffusion models on labelled anomaly periods.

- `nyc_taxi_test.csv`  
  - Subset of `nyc_taxi_full.csv` where `split == "test"`.  
  - Held-out data used for final evaluation and reporting of results in the thesis.

All three split-specific files are derived directly from `nyc_taxi_full.csv`, ensuring a consistent
and reproducible link between the full processed dataset and the train/validation/test segments used
in experiments.

